# Mushrooms

References:
- Kaggle [Mushroom Classification](https://www.kaggle.com/datasets/uciml/mushroom-classification/data)
- UCI ML [Mushroom](https://archive.ics.uci.edu/dataset/73/mushroom)
- Associazione Micologica Bresadola [Mushroom Toxicity](https://ambbresadola.it/tossicita-dei-funghi/)
- Other Works [Edibility Detection of Mushroom Using Ensemble Methods](https://www.mecs-press.org/ijigsp/ijigsp-v11-n4/IJIGSP-V11-N4-5.pdf)

### 1. Load Data and Decodification

In [138]:
import numpy as np
import polars as pl
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.stats import chi2_contingency
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import OrdinalEncoder

in_file = '/Users/marcomanduca/Desktop/data_science/machine_learning/exam/data/agaricus-lepiota.data'

In [2]:
df = pl.read_csv(
    in_file,
    has_header=False,
    null_values=['?']
)

df.columns = [
        'class', 'cap_shape', 'cap_surface', 'cap_color','bruises','odor','gill_attachment',
        'gill_spacing','gill_size','gill_color','stalk_shape','stalk_root','stalk_surface_above_ring',
        'stalk_surface_below_ring','stalk_color_above_ring','stalk_color_below_ring','veil_type','veil_color',
        'ring_number','ring_type','spore_print_color','population','habitat'
    ]

In [3]:
mappings = {
    ## CLASS FEATURE
    # edible or poisonous
    "class": {"e": "edible", "p": "poisonous"},
    
    ## CAP FEATURES
    # shape of the cap
    "cap_shape": {"b": "bell", "c": "conical", "x": "convex", "f": "flat", "k": "knobbed", "s": "sunken"},
    # cap surface texture and appearance
    "cap_surface": {"f": "fibrous", "g": "grooves", "y": "scaly", "s": "smooth"},
    # cap color
    "cap_color": {"n": "brown", "b": "buff", "c": "cinnamon", "g": "gray", "r": "green", "p": "pink", "u": "purple", "e": "red", "w": "white", "y": "yellow"},

    ## GILL (lamelle) FEATURES
    # how the gills are attached to the stalk
    "gill_attachment": {"a": "attached", "d": "descending", "f": "free", "n": "notched"},
    # gill spacing
    "gill_spacing": {"c": "close", "w": "crowded", "d": "distant"},
    # gill size
    "gill_size": {"b": "broad", "n": "narrow"},
    # gill color
    "gill_color": {"k": "black", "n": "brown", "b": "buff", "h": "chocolate", "g": "gray", "r": "green", "o": "orange", "p": "pink", "u": "purple", "e": "red", "w": "white", "y": "yellow"},

    ## STALK (gambo) FEATURES
    # stalk shape towards the base
    "stalk_shape": {"e": "enlarging", "t": "tapering"},
    # stalk root shape/type
    "stalk_root": {"b": "bulbous", "c": "club", "u": "cup", "e": "equal", "z": "rhizomorphs", "r": "rooted"},
    # stalk surface above the ring
    "stalk_surface_above_ring": {"f": "fibrous", "y": "scaly", "k": "silky", "s": "smooth"},
    # stalk surface below the ring
    "stalk_surface_below_ring": {"f": "fibrous", "y": "scaly", "k": "silky", "s": "smooth"},
    # stalk color above the ring
    "stalk_color_above_ring": {"n": "brown", "b": "buff", "c": "cinnamon", "g": "gray", "o": "orange", "p": "pink", "e": "red", "w": "white", "y": "yellow"},
    # stalk color below the ring
    "stalk_color_below_ring": {"n": "brown", "b": "buff", "c": "cinnamon", "g": "gray", "o": "orange", "p": "pink", "e": "red", "w": "white", "y": "yellow"},

    ## VEIL and RING FEATURES
    # veil type
    "veil_type": {"p": "partial", "u": "universal"},
    # veil color
    "veil_color": {"n": "brown", "o": "orange", "w": "white", "y": "yellow"},
    # number of rings
    "ring_number": {"n": "none", "o": "one", "t": "two"},
    # type of ring
    "ring_type": {"c": "cobwebby", "e": "evanescent", "f": "flaring", "l": "large", "n": "none", "p": "pendant", "s": "sheathing", "z": "zone"},

    ## OTHER PHYSICS FEATURES
    # change in cap color when bruised
    "bruises": {"t": "bruises", "f": "no"},
    # odor of the mushroom
    "odor": {"a": "almond", "l": "anise", "c": "creosote", "y": "fishy", "f": "foul", "m": "musty", "n": "none", "p": "pungent", "s": "spicy"},
    # color of the spore on a surface
    "spore_print_color": {"k": "black", "n": "brown", "b": "buff", "h": "chocolate", "r": "green", "o": "orange", "u": "purple", "w": "white", "y": "yellow"},

    ## ECOLOGICAL FEATURES
    # population size/growth pattern
    "population": {"a": "abundant", "c": "clustered", "n": "numerous", "s": "scattered", "v": "several", "y": "solitary"},
    # habitat where mushroom is found
    "habitat": {"g": "grasses", "l": "leaves", "m": "meadows", "p": "paths", "u": "urban", "w": "waste", "d": "woods"}
}

In [4]:
df = df.with_columns(
    [pl.col(col_name).replace(mapping) for col_name, mapping in mappings.items() if col_name in df.columns]
)

In [5]:
df.sample(10)

class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,stalk_shape,stalk_root,stalk_surface_above_ring,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""poisonous""","""flat""","""smooth""","""brown""","""no""","""spicy""","""free""","""close""","""narrow""","""buff""","""tapering""",null,"""silky""","""silky""","""white""","""pink""","""partial""","""white""","""one""","""evanescent""","""white""","""several""","""leaves"""
"""edible""","""convex""","""scaly""","""yellow""","""bruises""","""anise""","""free""","""close""","""broad""","""brown""","""enlarging""","""club""","""smooth""","""smooth""","""white""","""white""","""partial""","""white""","""one""","""pendant""","""black""","""numerous""","""grasses"""
"""edible""","""flat""","""fibrous""","""gray""","""bruises""","""none""","""free""","""close""","""broad""","""purple""","""tapering""","""bulbous""","""smooth""","""smooth""","""pink""","""gray""","""partial""","""white""","""one""","""pendant""","""black""","""solitary""","""woods"""
"""poisonous""","""flat""","""scaly""","""brown""","""no""","""fishy""","""free""","""close""","""narrow""","""buff""","""tapering""",null,"""silky""","""silky""","""white""","""pink""","""partial""","""white""","""one""","""evanescent""","""white""","""several""","""leaves"""
"""edible""","""flat""","""scaly""","""red""","""bruises""","""none""","""free""","""close""","""broad""","""white""","""tapering""","""bulbous""","""smooth""","""smooth""","""gray""","""white""","""partial""","""white""","""one""","""pendant""","""brown""","""solitary""","""woods"""
"""poisonous""","""flat""","""scaly""","""brown""","""bruises""","""pungent""","""free""","""close""","""narrow""","""pink""","""enlarging""","""equal""","""smooth""","""smooth""","""white""","""white""","""partial""","""white""","""one""","""pendant""","""black""","""several""","""grasses"""
"""poisonous""","""convex""","""smooth""","""red""","""no""","""fishy""","""free""","""close""","""narrow""","""buff""","""tapering""",null,"""silky""","""smooth""","""pink""","""pink""","""partial""","""white""","""one""","""evanescent""","""white""","""several""","""woods"""
"""poisonous""","""flat""","""smooth""","""pink""","""bruises""","""none""","""free""","""close""","""broad""","""gray""","""enlarging""","""bulbous""","""smooth""","""smooth""","""white""","""white""","""partial""","""white""","""two""","""pendant""","""green""","""several""","""grasses"""
"""edible""","""convex""","""scaly""","""brown""","""bruises""","""none""","""free""","""close""","""broad""","""white""","""tapering""","""bulbous""","""smooth""","""smooth""","""white""","""white""","""partial""","""white""","""one""","""pendant""","""black""","""solitary""","""woods"""


### 2. Descriptive Analysis

In [6]:
df.describe()
# stalk_root > 2480 missing values

statistic,class,cap_shape,cap_surface,cap_color,bruises,odor,gill_attachment,gill_spacing,gill_size,gill_color,stalk_shape,stalk_root,stalk_surface_above_ring,stalk_surface_below_ring,stalk_color_above_ring,stalk_color_below_ring,veil_type,veil_color,ring_number,ring_type,spore_print_color,population,habitat
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""5644""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124""","""8124"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""2480""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""edible""","""bell""","""fibrous""","""brown""","""bruises""","""almond""","""attached""","""close""","""broad""","""black""","""enlarging""","""bulbous""","""fibrous""","""fibrous""","""brown""","""brown""","""partial""","""brown""","""none""","""evanescent""","""black""","""abundant""","""grasses"""
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""poisonous""","""sunken""","""smooth""","""yellow""","""no""","""spicy""","""free""","""crowded""","""narrow""","""yellow""","""tapering""","""rooted""","""smooth""","""smooth""","""yellow""","""yellow""","""partial""","""yellow""","""two""","""pendant""","""yellow""","""solitary""","""woods"""


In [7]:
# Distribution of target classes
class_counts = df.select(pl.col("class").value_counts(sort=True)).unnest("class")

# Visualizzazione
fig = px.bar(
    class_counts,
    x="class",
    y="count",
    color="class",
    title="Class distribution of Mushrooms (Edible vs Poisonous)",
    color_discrete_map={'edible': '#00cc96', 'poisonous': '#aa63fa'})
fig.show()

In [8]:
for feature in df.columns:
    if feature != 'class':
        odor_analysis = (
            df.group_by([feature, "class"])
            .agg(pl.len().alias("count"))
        )
        fig = px.bar(
            odor_analysis,
            x=feature,
            y="count",
            color="class",
            barmode="group",
            title=f"Relationship between {feature.capitalize()} and Edibility",
            color_discrete_map={'edible': '#00cc96', 'poisonous': '#aa63fa'})
        fig.show()

In [9]:
odor_perc = (
    df.group_by(["odor", "class"])
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum().over("odor") * 100).alias("percentage")
    )
    .sort("odor")
)

print(odor_perc)

shape: (10, 4)
┌──────────┬───────────┬───────┬────────────┐
│ odor     ┆ class     ┆ count ┆ percentage │
│ ---      ┆ ---       ┆ ---   ┆ ---        │
│ str      ┆ str       ┆ u32   ┆ f64        │
╞══════════╪═══════════╪═══════╪════════════╡
│ almond   ┆ edible    ┆ 400   ┆ 100.0      │
│ anise    ┆ edible    ┆ 400   ┆ 100.0      │
│ creosote ┆ poisonous ┆ 192   ┆ 100.0      │
│ fishy    ┆ poisonous ┆ 576   ┆ 100.0      │
│ foul     ┆ poisonous ┆ 2160  ┆ 100.0      │
│ musty    ┆ poisonous ┆ 36    ┆ 100.0      │
│ none     ┆ edible    ┆ 3408  ┆ 96.598639  │
│ none     ┆ poisonous ┆ 120   ┆ 3.401361   │
│ pungent  ┆ poisonous ┆ 256   ┆ 100.0      │
│ spicy    ┆ poisonous ┆ 576   ┆ 100.0      │
└──────────┴───────────┴───────┴────────────┘


In [10]:
# Unique value for each columns
cardinality = df.select([
    pl.col(c).n_unique().alias(c) for c in df.columns
]).unpivot(variable_name="Feature", value_name="Unique_Values")

print(cardinality.sort("Unique_Values", descending=True))

shape: (23, 2)
┌────────────────────────┬───────────────┐
│ Feature                ┆ Unique_Values │
│ ---                    ┆ ---           │
│ str                    ┆ u32           │
╞════════════════════════╪═══════════════╡
│ gill_color             ┆ 12            │
│ cap_color              ┆ 10            │
│ odor                   ┆ 9             │
│ stalk_color_above_ring ┆ 9             │
│ stalk_color_below_ring ┆ 9             │
│ …                      ┆ …             │
│ gill_attachment        ┆ 2             │
│ gill_spacing           ┆ 2             │
│ gill_size              ┆ 2             │
│ stalk_shape            ┆ 2             │
│ veil_type              ┆ 1             │
└────────────────────────┴───────────────┘


In [11]:
# veil_type has only one unique value across the dataset, we can consider dropping it for further analysis.
df = df.drop(['veil_type'])

In [12]:
# Heatmap: Habitat vs Population
heatmap_data = (
    df.group_by(["habitat", "population", 'class'])
    .agg(pl.len().alias("count"))
)

fig = px.density_heatmap(
    heatmap_data,
    x="habitat", 
    y="population", 
    z="count",
    title="Heatmap: Habitat vs Population",
    text_auto=True,
)
fig.show()

### 3. Some Statistics

Cramer's V is a statistical index to measure the association between 2 nominal features (categorical), it's based on the Person's Chi-Squared index ($\chi^2$).

$V =\sqrt{ \frac{\chi^2}{n * \min(k-1, r-1)}} \in [0,1]$

Where:
- $\chi^2$ is the Chi-Squared value obtained by the contingency table
- n is the grand total of rows of the sample
- k is the number of column of the dataset
- r is the number of rows of the dataset

In [97]:
# Calculate Cramér's V for categorical features relative to the target class
# This will give us an idea of feature importance based on their association with the target variable.
def cramers_v_vs_target(df, target_col):
    results = []
    for feature in df.columns:
        if feature != target_col:
            double_entry_matrix = (
                df.select([feature, target_col])
                .pivot(index=feature, on=target_col, values=feature, aggregate_function='len')
                .drop(feature)
                .to_numpy()
            )
            chi2, p, dof, ex = chi2_contingency(double_entry_matrix)
            n = double_entry_matrix.sum()
            results.append({
                "feature": feature,
                "p_value": p,
                "significance": p < 0.05,
                "chi2_stat": chi2,
                "cramers_v": np.sqrt(chi2 / (n * (min(double_entry_matrix.shape) - 1)))
            })
    return pl.DataFrame(results).sort("cramers_v", descending=True)

variable_importance = cramers_v_vs_target(df, 'class')

In [123]:
fig = px.bar(
    variable_importance, 
    x="cramers_v", 
    y="feature", 
    orientation='h',
    title="Association (Cramér's V) between Features and Class",
    labels={"cramers_v": "Strength of Association", "feature": "Feature"},
    color="cramers_v"
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# Calculate Cramér's V correlation matrix for all couples of categorical features except the target class
# This will help us understand inter-feature relationships (correlations) between features to identify if some are redundant.
def cramers_v_corr(df, target_col):
    results = []
    for i, feat1 in enumerate(df.columns):
        for j, feat2 in enumerate(df.columns):
            if i < j and feat1 != target_col and feat2 != target_col:
                
                # Contingency table
                pivot_df = (
                    df.select([feat1, feat2])
                    .pivot(index=feat1, on=feat2, values=feat2, aggregate_function='len')
                    .fill_null(0)
                )
                
                # 1. Drop the index column to get the matrix
                matrix = pivot_df.drop(feat1).to_numpy()
                
                # 2. Clean the matrix: remove rows or columns with all zeros (this avoids "expected frequencies has a zero element" error)
                matrix = matrix[matrix.sum(axis=1) > 0]
                matrix = matrix[:, matrix.sum(axis=0) > 0]
                
                # If the matrix is too small after cleaning (e.g., 1x1), skip
                if matrix.shape[0] < 2 or matrix.shape[1] < 2:
                    continue
                
                try:
                    chi2, p, dof, ex = chi2_contingency(matrix)
                    n = matrix.sum()
                    v = np.sqrt(chi2 / (n * (min(matrix.shape) - 1)))
                    
                    results.append({
                        "feature_1": feat1,
                        "feature_2": feat2,
                        "p_value": p,
                        "cramers_v": v
                    })
                except ValueError:
                    continue
    return pl.DataFrame(results).sort("cramers_v", descending=True)

corr_matrix_df = cramers_v_corr(df, 'class')

shape: (210, 4)
┌────────────────────────┬────────────────────────┬────────────┬───────────┐
│ feature_1              ┆ feature_2              ┆ p_value    ┆ cramers_v │
│ ---                    ┆ ---                    ┆ ---        ┆ ---       │
│ str                    ┆ str                    ┆ f64        ┆ f64       │
╞════════════════════════╪════════════════════════╪════════════╪═══════════╡
│ gill_attachment        ┆ stalk_color_above_ring ┆ 0.0        ┆ 0.977755  │
│ gill_attachment        ┆ stalk_color_below_ring ┆ 0.0        ┆ 0.977755  │
│ gill_attachment        ┆ veil_color             ┆ 0.0        ┆ 0.955097  │
│ gill_attachment        ┆ spore_print_color      ┆ 0.0        ┆ 0.826898  │
│ stalk_color_above_ring ┆ veil_color             ┆ 0.0        ┆ 0.816497  │
│ …                      ┆ …                      ┆ …          ┆ …         │
│ gill_spacing           ┆ stalk_shape            ┆ 3.8439e-13 ┆ 0.080558  │
│ cap_surface            ┆ stalk_shape            ┆ 6.5428e-

In [110]:
## Heatmap Visualization of Cramér's V Correlation Matrix

# 1. List of all features involved
all_features = [col for col in df.columns if col != 'class']

# 2. Create the diagonal data (self-correlation = 1)
diag_data = pl.DataFrame({
    "feature_1": all_features,
    "feature_2": all_features,
    "cramers_v": [1.0] * len(all_features)
})

# 3. Create the mirrored data (feature_2, feature_1)
mirror_data = corr_matrix_df.select([
    pl.col("feature_2").alias("feature_1"),
    pl.col("feature_1").alias("feature_2"),
    pl.col("cramers_v")
])

# 4. Combine original, mirrored, and diagonal data
full_matrix_df = pl.concat([
    corr_matrix_df.select(["feature_1", "feature_2", "cramers_v"]),
    mirror_data,
    diag_data
]).unique(subset=["feature_1", "feature_2"]) # Avoid duplicates if the diagonal already exists

# 5. Pivot to create the matrix structure
heatmap_pivot = (
    full_matrix_df.pivot(index="feature_1", on="feature_2", values="cramers_v")
    .fill_null(0) # Pairs not calculated or removed due to statistical errors become 0
    .sort("feature_1") # Sort rows alphabetically
)

# Also reorder columns to have a perfectly symmetric matrix
col_order = ["feature_1"] + sorted(all_features)
heatmap_pivot = heatmap_pivot.select(col_order)

# 6. Plot
z_data = heatmap_pivot.drop("feature_1").to_numpy()
labels = heatmap_pivot["feature_1"].to_list()

fig = px.imshow(
    z_data,
    x=labels,
    y=labels,
    color_continuous_scale="Viridis",
    zmin=0, zmax=1, # Fix the scale from 0 to 1
    title="Cramér's V Correlation Matrix (with Diagonal)"
)
fig.update_layout(width=900, height=800)
fig.show()

### 4. Clustering

Clustering with K-Medoids using Weighted Hamming Distance based on Cramér's V (without target label)

In [ ]:
encoder = OrdinalEncoder()
data_encoded = encoder.fit_transform(df.drop("class").to_pandas())

# Create weights based on Cramér's V
weights = variable_importance.filter(pl.col("feature").is_in(df.drop("class").columns))["cramers_v"].to_numpy()

# Function to calculate weighted Hamming distance between two rows
def weighted_hamming(x, y, w):
    return np.sum((x != y) * w)

# Calculate the distance matrix (N x N)
dist_matrix = pairwise_distances(data_encoded, metric=lambda x, y: weighted_hamming(x, y, weights))

In [ ]:
# Fit K-Medoids on the precomputed distance matrix
kmedoids_weighted = KMedoids(
    n_clusters=8,
    metric='precomputed',
    random_state=42,
    method='pam'
)

cluster_labels_weighted = kmedoids_weighted.fit_predict(dist_matrix)

df_clustered = df.with_columns(
    pl.Series("cluster", cluster_labels_weighted).cast(pl.Utf8)
)

In [161]:
feature = 'cluster'
analysis = (
    df_clustered.group_by([feature, "class"])
    .agg(pl.len().alias("count"))
)
fig = px.bar(
    analysis,
    x=feature,
    y="count",
    color="class",
    barmode="group",
    title=f"Relationship between {feature.capitalize()} and Edibility",
    color_discrete_map={'edible': '#00cc96', 'poisonous': '#aa63fa'})
fig.show()

In [166]:
from sklearn.metrics import silhouette_score

# Calcolo dello score sulla matrice di distanza pesata
score_weighted = silhouette_score(dist_matrix, cluster_labels_weighted, metric='precomputed')

print(f"Silhouette Score (Pesato): {score_weighted:.4f}")

Silhouette Score (Pesato): 0.2060


In [168]:
# Rieseguiamo il clustering con K=8 per consolidare i risultati
kmedoids_final = KMedoids(n_clusters=8, metric='precomputed', random_state=42)
cluster_labels_8 = kmedoids_final.fit_predict(dist_matrix)

df_final = df_clustered.with_columns(
    pl.Series("cluster_8", cluster_labels_8).cast(pl.Utf8)
)

# Calcoliamo la purezza per ogni cluster
purity_8 = (
    df_final.group_by("cluster_8")
    .agg([
        pl.len().alias("totale"),
        (pl.col("class") == "edible").sum().alias("commestibili"),
        (pl.col("class") == "poisonous").sum().alias("velenosi"),
        # Calcoliamo la percentuale della classe dominante
        (pl.col("class").value_counts().struct.field("count").max() / pl.len()).alias("purezza")
    ])
    .sort("cluster_8")
)

print(purity_8)

shape: (8, 5)
┌───────────┬────────┬──────────────┬──────────┬──────────┐
│ cluster_8 ┆ totale ┆ commestibili ┆ velenosi ┆ purezza  │
│ ---       ┆ ---    ┆ ---          ┆ ---      ┆ ---      │
│ str       ┆ u32    ┆ u32          ┆ u32      ┆ f64      │
╞═══════════╪════════╪══════════════╪══════════╪══════════╡
│ 0         ┆ 2103   ┆ 260          ┆ 1843     ┆ 0.876367 │
│ 1         ┆ 725    ┆ 594          ┆ 131      ┆ 0.81931  │
│ 2         ┆ 891    ┆ 740          ┆ 151      ┆ 0.830527 │
│ 3         ┆ 890    ┆ 726          ┆ 164      ┆ 0.81573  │
│ 4         ┆ 755    ┆ 636          ┆ 119      ┆ 0.842384 │
│ 5         ┆ 473    ┆ 450          ┆ 23       ┆ 0.951374 │
│ 6         ┆ 1362   ┆ 13           ┆ 1349     ┆ 0.990455 │
│ 7         ┆ 925    ┆ 789          ┆ 136      ┆ 0.852973 │
└───────────┴────────┴──────────────┴──────────┴──────────┘


In [169]:
# Analisi comparativa dei profili dominanti
focus_clusters = ["3", "6"]

identikit_comparativo = (
    df_final.filter(pl.col("cluster_8").is_in(focus_clusters))
    .group_by("cluster_8")
    .agg([
        pl.col("class").mode().first().alias("Classe Prev."),
        pl.col("odor").mode().first().alias("Odore"),
        pl.col("gill_size").mode().first().alias("Lamelle"),
        pl.col("spore_print_color").mode().first().alias("Colore Spore"),
        pl.col("stalk_root").mode().first().alias("Radice")
    ])
)

print(identikit_comparativo)

shape: (2, 6)
┌───────────┬──────────────┬───────┬─────────┬──────────────┬─────────┐
│ cluster_8 ┆ Classe Prev. ┆ Odore ┆ Lamelle ┆ Colore Spore ┆ Radice  │
│ ---       ┆ ---          ┆ ---   ┆ ---     ┆ ---          ┆ ---     │
│ str       ┆ str          ┆ str   ┆ str     ┆ str          ┆ str     │
╞═══════════╪══════════════╪═══════╪═════════╪══════════════╪═════════╡
│ 3         ┆ edible       ┆ none  ┆ broad   ┆ brown        ┆ bulbous │
│ 6         ┆ poisonous    ┆ foul  ┆ broad   ┆ chocolate    ┆ bulbous │
└───────────┴──────────────┴───────┴─────────┴──────────────┴─────────┘


In [170]:
# Isoliamo le eccezioni del Cluster 6
exceptions_c6 = df_final.filter(
    (pl.col("cluster_8") == "6") & (pl.col("class") == "edible")
)

# Vediamo cosa li rende diversi dai velenosi dello stesso cluster
# Confrontiamo ad esempio il colore del cappello o l'habitat
print("Caratteristiche dei 13 funghi commestibili nel Cluster 6:")
print(exceptions_c6.select(["odor", "spore_print_color", "stalk_surface_above_ring", "habitat", "cap_color"]).unique())

Caratteristiche dei 13 funghi commestibili nel Cluster 6:
shape: (5, 5)
┌──────┬───────────────────┬──────────────────────────┬─────────┬───────────┐
│ odor ┆ spore_print_color ┆ stalk_surface_above_ring ┆ habitat ┆ cap_color │
│ ---  ┆ ---               ┆ ---                      ┆ ---     ┆ ---       │
│ str  ┆ str               ┆ str                      ┆ str     ┆ str       │
╞══════╪═══════════════════╪══════════════════════════╪═════════╪═══════════╡
│ none ┆ white             ┆ smooth                   ┆ leaves  ┆ cinnamon  │
│ none ┆ white             ┆ scaly                    ┆ woods   ┆ brown     │
│ none ┆ white             ┆ scaly                    ┆ paths   ┆ brown     │
│ none ┆ white             ┆ fibrous                  ┆ leaves  ┆ cinnamon  │
│ none ┆ white             ┆ silky                    ┆ grasses ┆ white     │
└──────┴───────────────────┴──────────────────────────┴─────────┴───────────┘


In [174]:
def find_optimal_kmedoids(dist_matrix, k_range):
    best_k = -1
    best_score = -1
    best_labels = None
    results = []

    print("Ricerca del numero ottimale di cluster...")
    
    for k in k_range:
        # Inizializzazione e fit
        km = KMedoids(
            n_clusters=k,
            metric='precomputed',
            random_state=42,
            init='k-medoids++',
            max_iter=1_000,
            method='pam'
        )
        labels = km.fit_predict(dist_matrix)
        
        # Calcolo Silhouette
        score = silhouette_score(dist_matrix, labels, metric='precomputed')
        results.append((k, score))
        
        print(f"K={k} | Silhouette Score: {score:.4f}")
        
        # Aggiornamento del modello migliore
        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels
            
    print(f"\nConfigurazione ottimale trovata: K={best_k} con Score={best_score:.4f}")
    return best_k, best_labels, results

# 1. Definiamo il range di test (es. da 2 a 12)
k_values = range(6, 10)

# 2. Eseguiamo l'ottimizzazione
best_k, best_labels, all_results = find_optimal_kmedoids(dist_matrix, k_values)

# 3. Aggiorniamo il DataFrame con i parametri ottimali
df_clustered = df.with_columns(
    pl.Series("cluster", best_labels).cast(pl.Utf8)
)

Ricerca del numero ottimale di cluster...


/Users/marcomanduca/miniconda3/envs/ai/lib/python3.13/site-packages/sklearn/utils/deprecation.py:95: FutureWarning:

Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.



KeyboardInterrupt: 